# 📈 S&P 500 Pairs Trading 交易期 (Trading Period) 策略邏輯與公式詳解

## 📝 概述
在配對交易中，**交易期 (Trading Period)**（預設 $T = 126$ 天）的核心任務是**根據 Spread Z-Score 訊號執行交易，並結合多層風控管理資金**。

> [!IMPORTANT]
> 本文件以 `strategies/trading/` 下實際運行的 `.py` 原始碼為唯一依據。

### 策略池（`config.py` `strategies_raw_all`，共 13 個）：

| # | 策略 | 形成期模組 | 交易期模組 |
| :---: | :--- | :--- | :--- |
| 1 | SSD Basic | `ssd_basic.py` | `zscore_trading.py` |
| 2 | SSD Rolling | `ssd_rolling.py` | `zscore_trading.py` |
| 3 | DTW Paper (DTW) | `DTW_Cointegration_Paper.py` | `zscore_trading.py` |
| 4 | DTW Paper (SSD-DTW-PCA) | `DTW_Cointegration_Paper.py` | `zscore_trading.py` |
| 5 | HDBSCAN MultiScale | `HDBSCAN_MultiScale.py` | `zscore_trading.py` |
| 6 | HDBSCAN MultiScale PCA-UMAP | `HDBSCAN_MultiScale.py` | `zscore_trading.py` |
| 7 | HDBSCAN UMAP | `HDBSCAN_UMAP.py` | `zscore_trading.py` |
| 8 | HDBSCAN UMAP PCA-UMAP | `HDBSCAN_UMAP.py` | `zscore_trading.py` |
| 9 | Ensemble HDBSCAN | `ensemble.py` | `zscore_trading.py` |
| 10 | Ensemble SSD-DTW | `ensemble.py` | `zscore_trading.py` |
| 11 | SSD Rolling DRL | `ssd_rolling.py` | `drl_lstm_trading.py` |
| 12 | HDBSCAN UMAP DRL | `HDBSCAN_UMAP.py` | `drl_lstm_trading.py` |
| 13 | HDBSCAN MultiScale DRL | `HDBSCAN_MultiScale.py` | `drl_lstm_trading.py` |

`strategies_raw` 決定實際執行範圍（如 `strategies_raw_all[-3:]` 只跑 #11–#13，`strategies_raw_all` 跑全部 13 個）。

### 📂 交易期模組架構：

```
drl_lstm_trading.py          ← #11–#13 使用（DRL 強化學習）
    繼承/匯入 zscore_trading.py  ← 基礎 Z-Score 狀態機，提供 Spread 計算邏輯

zscore_trading.py            ← #1–#10 使用（通用 Z-Score 狀態機）
pure_dtw_trading.py          ← 純 DTW 交叉帶內進場（程式碼保留，目前無策略使用）
```

### 回測參數（`config.py` `base_params`）：

| 參數 | 值 |
| :--- | :--- |
| `entry_z` | 2.0（進場閾值） |
| `exit_z` | 0.0（出場閾值，回歸至均值） |
| `max_holding_days` | 30 天（超時強制平倉） |
| `fee_rate` + `slippage_rate` | 各 0.001（單程各 0.1%，來回總摩擦 0.4%） |
| `zscore_window` | 0（靜態形成期參數，不滾動更新） |
| `stop_loss_pct` | 0.0（預設不啟用個對停損） |
| `portfolio_stop_loss_pct` | 0.0（預設不啟用全局停損） |
| `use_vol_adjust` | False（預設不啟用波動率自適應調節） |

## 🧠 一、 Z-Score 狀態機核心邏輯 (`zscore_trading.py`)

所有策略的 Spread 重建與 Z-Score 計算均通過此模組完成。根據形成期存入的欄位，自動路由到對應的計算空間。

### 1.1 Spread 重建三條路徑

**路徑 A — 原始 log-price OLS 殘差空間**（當 `OLS_Alpha` 不為 None 時觸發）

適用策略：DTW、HDBSCAN UMAP、HDBSCAN MultiScale

$$\text{Spread}_t = \ln P_{A,t} - \alpha - \beta \cdot \ln P_{B,t}$$

其中 $\alpha$、$\beta$ 為形成期 OLS 迴歸的截距與斜率，在整個交易期保持不變（靜態模式 `zscore_window=0`）。

**路徑 B1 — 累積回報比值空間**（當 `OLS_Alpha` 為 None 且 `First_Price_A/B > 0` 時）

適用策略：SSD Basic

$$\text{Spread}_t = \frac{P_{A,t}}{P_{A,0}} - \frac{P_{B,t}}{P_{B,0}}$$

**路徑 B2 — Z-Score 標準化對數空間**（當 `OLS_Alpha` 為 None 且無 `First_Price` 時）

適用策略：SSD Rolling、Ensemble SSD-DTW

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}^{form}}{\sigma_{\ln P_i}^{form}}, \quad \text{Spread}_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

> **三條路徑各自使用對應的 `Spread_Mean`/`Spread_Std`**（形成期在同一空間內計算並存入 `Formation_Params`），確保 Z-Score 基準一致。

### 1.2 Z-Score 計算（靜態模式 `zscore_window=0`）

$$Z_t = \text{clip}\left(\frac{\text{Spread}_t - \mu_\epsilon}{\max(\sigma_\epsilon,\, \sigma_\text{min})},\; -10,\; 10\right)$$

若啟用波動率自適應（`use_vol_adjust=True`）：

$$\sigma_\text{adj} = \max\!\left(\sigma_\epsilon \cdot \max\!\left(1,\, \frac{\sigma_{20}}{\sigma_\epsilon}\right),\; \sigma_\text{min}\right), \quad Z_t = \text{clip}\left(\frac{\text{Spread}_t - \mu_\epsilon}{\sigma_\text{adj}},\; -10,\; 10\right)$$

### 1.3 進出場訊號（Z-Score 狀態機）

| 條件 | 動作 |
| :--- | :--- |
| $Z_t > \text{entry\_z}$（= 2.0） | **空頭建倉**：空 Ticker\_A，多 Ticker\_B |
| $Z_t < -\text{entry\_z}$ | **多頭建倉**：多 Ticker\_A，空 Ticker\_B |
| $Z_t \le \text{exit\_z}$（= 0.0）且原為空頭 | **平倉** |
| $Z_t \ge -\text{exit\_z}$ 且原為多頭 | **平倉** |
| 持倉超過 `max_holding_days`（= 30）天 | **超時強制平倉** |

### 1.4 資金部位配置（風險中性加權）

$$W_{total} = 1.0 + |\beta|, \quad v_A = C_{pair} \cdot \frac{1.0}{W_{total}}, \quad v_B = C_{pair} \cdot \frac{|\beta|}{W_{total}}$$

- 多頭方向：買 $v_A$ 的 Ticker\_A，賣 $v_B$ 的 Ticker\_B
- 空頭方向：賣 $v_A$ 的 Ticker\_A，買 $v_B$ 的 Ticker\_B
- 當 $\beta = 1.0$（SSD Basic）退化為 50%/50% 等市值配置

### 1.5 PnL 計算

$$\text{Raw Unrealized}_t = n_A \cdot (P_{A,t} - P_{A,\text{entry}}) + n_B \cdot (P_{B,t} - P_{B,\text{entry}})$$
$$\text{Entry Fee} = (|n_A| P_{A,\text{entry}} + |n_B| P_{B,\text{entry}}) \cdot f_\text{total}$$
$$\text{Exit Fee} = (|n_A| P_{A,t} + |n_B| P_{B,t}) \cdot f_\text{total}, \quad f_\text{total} = \text{fee\_rate} + \text{slippage\_rate}$$
$$\text{Trade PnL} = \text{Raw Unrealized} - \text{Entry Fee} - \text{Exit Fee}$$

## ⏳ 二、 純 DTW 交叉帶內進場邏輯 (`pure_dtw_trading.py`)

> **目前無策略使用此模組**（config.py 啟用的三個策略均使用 `drl_lstm_trading.py`），但程式碼保留備用。

### 2.1 Spread 重建（與 zscore_trading 路徑 A 一致）

pure_dtw_trading 繼承 `zscore_trading.Trading` 並覆寫 `_simulate_pair`。Spread 的計算空間已與形成期 DTW_Cointegration_Paper 對齊：

- 當 `ols_alpha` 不為 None（DTW 形成期傳入）：
$$\text{Spread}_t = \ln P_{A,t} - \alpha - \beta \cdot \ln P_{B,t}$$
- 當 `ols_alpha` 為 None（Fallback）：
$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_i}{\sigma_i}, \quad \text{Spread}_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

### 2.2 交叉帶內進場條件（Cross Back Inside the Bands）

傳統 Z-Score 策略在「突破」臨界線時立刻進場（容易遇到持續發散風險）。DTW 策略要求 Z-Score **先突破後折返穿越**才建倉：

| 訊號 | 條件 |
| :--- | :--- |
| 空頭建倉 | $Z_{t-1} > \text{entry\_z}$ 且 $Z_t \le \text{entry\_z}$（從上方折返穿越） |
| 多頭建倉 | $Z_{t-1} < -\text{entry\_z}$ 且 $Z_t \ge -\text{entry\_z}$（從下方折返穿越） |

### 2.3 冷卻期重設條件

停損後進入方向性冷卻（`cooldown_dir`），需等 Z-Score 回穿 0 才能解凍：

- 多頭停損後（`cooldown_dir = 1`）：等待 $Z_t \le 0.0$
- 空頭停損後（`cooldown_dir = -1`）：等待 $Z_t \ge 0.0$

> 此設計確保 Spread 完全回歸中性後才允許再次進場，避免在發散趨勢中連續停損。

### 2.4 部位配置

固定 $\beta = 1.0$（等市值），資金 50%/50% 分配。

## 🤖 三、 DRL LSTM 交易期邏輯 (`drl_lstm_trading.py`)

> **目前三個啟用策略均使用此模組**（SSD Rolling DRL、HDBSCAN UMAP DRL、HDBSCAN MultiScale DRL）。

此策略將每個配對的交易問題建模為**馬可夫決策過程 (MDP)**，在形成期資料上訓練 LSTM-DQN 代理人，再於交易期推論最佳進出場動作。

### 3.1 訓練流程（每配對每期獨立）

```
對每一個配對 (Ticker_A, Ticker_B) 在每一個滾動期：
  1. 從 formation_start 到 formation_end 切出形成期價格
  2. 計算 8 維特徵序列（見 3.3）
  3. 以 Gymnasium 環境包裝，進行 drl_episodes = 40 輪 DQN 訓練
  4. Agent 快取於 Trading._shared_agents[f"{period_start}_{trade_start}_{A}_{B}"]
  5. 同一配對同一期的後續呼叫直接取快取（不重複訓練）
```

> **注意**：每個配對各自訓練獨立的 agent，並非跨配對共享。快取 key 包含 ticker 名稱，不同配對得到不同 agent。

### 3.2 Spread 計算（同 zscore_trading 邏輯）

在形成期與交易期均使用形成期傳入的 `Log_Mean_A/B`、`Log_Std_A/B` 進行 Z-Score 標準化：

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_i^{form}}{\sigma_i^{form}}, \quad \text{Spread}_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

$$Z_t = \frac{\text{Spread}_t - \mu_\epsilon}{\sigma_\epsilon}$$

### 3.3 Gymnasium 環境特徵空間（8 維觀測向量）

| 維度 | 特徵名稱 | 計算方式 |
| :---: | :--- | :--- |
| 1 | `ZScore` | 即時 Spread Z-Score $Z_t$ |
| 2 | `Rel_Return` | 兩股日收益率差：$R_{A,t} - R_{B,t}$ |
| 3 | `MA_Dist` | Spread 5 日均線與 21 日均線的差值（短期動量） |
| 4 | `time_to_maturity` | 剩餘交易日比例：$(T - t) / T$ |
| 5 | `Spread_Std` | 20 日滾動 Spread 標準差（相對歸一化） |
| 6 | `position` | 當前倉位方向：$\{-1, 0, 1\}$ |
| 7 | `days_held_norm` | 持倉天數 / 交易期總天數 |
| 8 | `Spread_Trend` | $Z_t - Z_{t-5}$（5 日 Z-Score 動量，clip $\pm 5$） |

### 3.4 動作空間

| action\_idx | 動作 | 說明 |
| :---: | :--- | :--- |
| 0 | 平倉 / 空倉 | 強制平倉已有部位，或保持無倉 |
| 1 | Long Spread | 買 Ticker\_A，賣 Ticker\_B（看多 Spread） |
| 2 | Short Spread | 賣 Ticker\_A，買 Ticker\_B（看空 Spread） |

### 3.5 獎勵函數

$$R_t = R_\text{close} + R_\text{entry\_penalty} + R_\text{daily\_hold} + R_\text{forced\_close}$$

| 獎勵項 | 公式 | 觸發時機 |
| :--- | :--- | :--- |
| 平倉收益 $R_\text{close}$ | $\dfrac{\text{Trade PnL}}{C_{pair}} \times 100$ | 動作改變且原有部位 |
| 開倉摩擦懲罰 | $-\dfrac{\text{Entry Fee}}{C_{pair}} \times 100 \times 0.5$ | 開倉時 |
| 日持倉獎勵 | $\dfrac{n_A(P_{A,t+1}-P_{A,t}) + n_B(P_{B,t+1}-P_{B,t})}{C_{pair}} \times 100 \times 0.3$ | 每日持倉（用次日價格計算） |
| 強制平倉 | $\dfrac{\text{Trade PnL}}{C_{pair}} \times 100$ | 到達 max\_steps 時 |

### 3.6 LSTM-DQN 網路架構

```
輸入 → LSTM（seq_len=10 步, hidden_dim=64, num_layers=1）→ 全連接層 → 3 個 Q 值
```

訓練使用 Experience Replay（`memory=deque(maxlen=10000)`），每 4 步更新一次，每 5 輪同步 target network。$\epsilon$-greedy 探索從 1.0 衰減至 0.05。

### 3.7 推論（交易期）

```
對每個交易日 t：
  1. 計算 8 維觀測 obs_t
  2. 維護最近 10 步的狀態序列 state_seq（deque maxlen=10）
  3. agent.model.eval()；以 argmax Q 值選擇動作（無隨機探索）
  4. 依動作執行平/開倉，記錄 PnL
```

In [1]:
# drl_lstm_trading.py — LSTM-DQN 網路與 8 維觀測向量實作示範
import torch            # type: ignore
import torch.nn as nn   # type: ignore
import numpy as np      # type: ignore

class LSTM_DQN(nn.Module):
    """LSTM-based Deep Q-Network for pairs trading."""
    def __init__(self, input_dim=8, hidden_dim=64, output_dim=3, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x shape: (batch, seq_len=10, input_dim=8)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # 取最後一步的 hidden state → 3 個 Q 值

# ── 8 維觀測向量計算示範 ────────────────────────────────────────────────────
def build_observation(zscore_series, price_a, price_b, t, max_steps, position, days_held):
    """
    返回 8 維 obs_t，對應 PairTradingEnv._get_obs() 的實際邏輯。
    """
    z      = float(zscore_series[t])
    z_prev = float(zscore_series[t - 5]) if t >= 5 else float(zscore_series[0])

    ret_a = (price_a[t] / price_a[t-1] - 1) if t > 0 else 0.0
    ret_b = (price_b[t] / price_b[t-1] - 1) if t > 0 else 0.0
    rel_return = float(ret_a - ret_b)

    ma5  = float(np.mean(zscore_series[max(0, t-4):t+1]))
    ma21 = float(np.mean(zscore_series[max(0, t-20):t+1]))
    ma_dist = ma5 - ma21

    ttm = (max_steps - t) / max_steps

    window = zscore_series[max(0, t-19):t+1]
    spread_std = float(np.std(window)) if len(window) > 1 else 1.0

    spread_trend = float(np.clip(z - z_prev, -5, 5))

    return np.array([
        z,                         # 1. ZScore
        rel_return,                # 2. Rel_Return
        ma_dist,                   # 3. MA_Dist (5日 - 21日)
        ttm,                       # 4. time_to_maturity
        spread_std,                # 5. Spread_Std (20日滾動)
        float(position),           # 6. position ∈ {-1, 0, 1}
        days_held / max_steps,     # 7. days_held_norm
        spread_trend,              # 8. Spread_Trend (Z_t - Z_{t-5}, clip ±5)
    ], dtype=np.float32)

# 訓練超參數
print("LSTM-DQN input_dim = 8, seq_len = 10, hidden_dim = 64")
print("Training: 40 episodes per pair-period, update every 4 steps, target sync every 5 ep.")
print("ε-greedy: 1.0 → 0.05 (decay 0.995 per episode)")

LSTM-DQN input_dim = 8, seq_len = 10, hidden_dim = 64
Training: 40 episodes per pair-period, update every 4 steps, target sync every 5 ep.
ε-greedy: 1.0 → 0.05 (decay 0.995 per episode)


## 🛡️ 四、 共享部位管理與六大風控機制 (`zscore_trading.py`)

所有交易模組均繼承 `zscore_trading.Trading`，共享以下六大風控防線：

### 風控機制一覽

| # | 機制名稱 | 觸發條件 | 行為 |
| :---: | :--- | :--- | :--- |
| 1 | **個股停損 SL** | 個配對未實現虧損 / $C_{pair}$ ≥ `stop_loss_pct`（預設停用=0） | 立即強平該配對，進入冷卻 |
| 2 | **動態 Z 發散停損 DSZ** | $\|Z_t\| >$ `dynamic_stop_z`（如 3.0 或 5.0） | 判定結構性破裂，強平並凍結 |
| 3 | **全域組合停損 PSL** | 總未實現虧損 / 初始資金 ≥ `portfolio_stop_loss_pct` | 斬倉**所有**持倉配對，重置資金計數 |
| 4 | **產業分散上限 MSR** | 單一產業配對數超過 $\max(1, \lfloor N \times \text{max\_sector\_ratio} \rfloor)$ | 超額配對排隊不進場 |
| 5 | **方向冷卻 Cooldown** | 停損後或強平後 | 等 Z-Score 穿越 0 才解凍（`cooldown_dir` 機制） |
| 6 | **波動率自適應 VOL ADJ** | `use_vol_adjust=True` | 以 20 日滾動 $\sigma$ 放大基準 $\sigma_{form}$，過濾無序震盪市場 |

### PSL（全域組合停損）正確 PnL 計帳邏輯

PSL 觸發時，開放部位尚未平倉，`Trade_PnL = 0`（僅在平倉時記帳）。因此必須使用 `Unrealized_PnL` 計算最終虧損：

$$\text{final\_realized} = \text{Realized\_PnL}_\text{before\_stop} + \text{Unrealized\_PnL}_\text{at\_stop}$$
$$\text{Trade\_PnL} = \text{Unrealized\_PnL}_\text{at\_stop} \quad \text{（平倉時 book 進去）}$$

> 若錯誤使用 `Trade_PnL`（平倉前為 0），PSL 觸發時開放部位的損失將永遠不被記帳，造成資金計算失真。

### 波動率自適應公式

$$\sigma_{adj} = \max\!\left(\sigma_{form} \times \max\!\left(1.0,\; \frac{\sigma_{roll20}}{\sigma_{form}}\right),\; \sigma_{\min}\right)$$

$$Z_t = \text{clip}\!\left(\frac{\text{Spread}_t - \mu_{form}}{\sigma_{adj}},\; -10,\; 10\right)$$

近期波動率顯著高於形成期時，進場 Z-Score 閾值被放大，有效降低震盪市場中的過度交易頻率。

## 📊 五、 所有交易期模組特徵對比總結

| 交易特徵 | Z-Score 狀態機 (`zscore_trading`) | 純 DTW 交叉進場 (`pure_dtw_trading`) | DRL LSTM (`drl_lstm_trading`) |
| :--- | :---: | :---: | :---: |
| **使用策略** | #1–#10（SSD/DTW/HDBSCAN/Ensemble 共 10 個） | — （程式碼保留，無策略使用） | #11–#13（SSD Rolling DRL、HDBSCAN UMAP DRL、HDBSCAN MultiScale DRL） |
| **開倉條件** | $\|Z_t\| >$ `entry_z` = 2.0 突破建倉 | Z-Score 從帶外折返穿越 `entry_z` | DRL agent 輸出動作 1 或 2 |
| **平倉條件** | $\|Z_t\| \le$ `exit_z` = 0.0 回歸均值 | Z-Score 回歸至均值中心 | DRL agent 輸出動作 0 (Flat) |
| **對沖比例** | OLS $\beta$ 風險中性加權 | 固定 $\beta = 1.0$（50%/50%） | OLS $\beta$ 風險中性加權 |
| **狀態特徵維度** | 1 維（$Z_t$） | 1 維（$Z_t$, 含前值 $Z_{t-1}$） | **8 維**（ZScore, Rel\_Return, MA\_Dist, TTM, Spread\_Std, position, days\_held\_norm, Spread\_Trend） |
| **時序記憶** | 無 | 兩日比較 | LSTM seq\_len = 10 天 |
| **訓練方式** | 無（規則型） | 無（規則型） | 每配對每期獨立訓練 40 episodes |
| **風控支援** | SL / DSZ / PSL / MSR / Cooldown / VOL ADJ | SL / DSZ / PSL / MSR / Cooldown / VOL ADJ | Env 內摩擦懲罰 + 強平機制；外層 PSL / MSR / DSZ 同樣適用 |

### Spread 空間對應關係

| 交易期模組 | Spread 空間 | 與形成期一致性 |
| :--- | :--- | :--- |
| `zscore_trading` Path A | OLS 殘差 log-price 空間 | 與 DTW/HDBSCAN 形成期一致 ✅ |
| `zscore_trading` Path B1 | 累積回報比值空間 | 與 SSD Basic 形成期一致 ✅ |
| `zscore_trading` Path B2 | Z-Score 標準化 log-price 空間 | 與 SSD Rolling 形成期一致 ✅ |
| `pure_dtw_trading` | OLS 殘差 log-price 空間（`ols_alpha` 路由） | 與 DTW 形成期一致 ✅ |
| `drl_lstm_trading` | 繼承 `zscore_trading` 路由邏輯 | 與所有形成期一致 ✅ |